# Manage Inference Engine Models and Deployments with the Python SDK

## Why This Matters
- Standardizes model, secret, deployment, job-monitoring, and cleanup workflows for Teradata Inference Engine workspaces.
- Keeps environment, authentication, and SDK-client setup in one repeatable setup section before any resource changes are made.
- Uses direct SDK calls and clear step-by-step cells so each operation is easy to inspect, troubleshoot, and rerun.


## What You Will Accomplish
- Configure an Inference Engine SDK client for an on-premises or Artemis workspace.
- Create and inspect secrets, models, model versions, deployments, jobs, and health checks through the Python SDK.
- Download and deploy an S3-backed model, then clean up demo resources through isolated notebook steps.


---

## Table of Contents

### [1. Setup & Prerequisites](#setup)
- [1.1 Imports](#imports)
- [1.2 Environment Configuration](#environment-configuration)
- [1.3 Authentication](#authentication)
- [1.4 Client Initialization](#client-initialization)
- [1.5 Reusable Helpers](#reusable-helpers)

### [2. Secrets APIs](#secrets)
- [2.1 List Workspace Secrets](#)
- [2.2 Create a HuggingFace Secret](#)
- [2.3 Inspect, Update, Rotate, and Delete Secrets](#)

### [3. Model Catalog APIs](#models)
- [3.1 List Workspace Models and Create a Model](#)
- [3.2 Inspect and Update the Selected Model](#)
- [3.3 Manage Model Versions](#model-versions)

### [4. Deployment APIs](#deployments)
- [4.1 List and Create Deployments](#)
- [4.2 Inspect Deployment Status](#)
- [4.3 Update and Delete Deployments](#)

### [5. Job Monitoring APIs](#jobs)
- [5.1 List and Sync Jobs](#)
- [5.2 Inspect a Specific Job](#)

### [6. Health APIs](#health)
- [6.1 Run Service Health Checks](#)

### [7. S3 Model Download and Deployment](#s3-model-workflow)
- [7.1 Configure S3 Model Metadata](#)
- [7.2 Create and Poll the S3 Model Download](#)
- [7.3 Deploy and Inspect the S3 Model](#)
- [7.4 Clean Up S3 Demo Resources](#)

### [8. Cleanup and Results](#cleanup)
- [8.1 Delete Demo Resources](#)
- [8.2 Results and Interpretation](#)
- [8.3 Summary / Next Steps](#)
---


<a id="setup"></a>
## 1. Setup & Prerequisites

Run these cells in order to load imports, capture environment-specific settings, create the bearer-auth object, initialize SDK clients, and define the reusable output helper used throughout the notebook.

> **Note:** This notebook is valid for on-premises Teradata deployments and Artemis environments only.


<a id="imports"></a>
### 1.1 Imports

This step imports the SDK client, service wrappers, request models, polling support, and formatted output helpers used by the notebook.


In [ ]:
# Import secure prompt handling, polling support, and formatted output.
from getpass import getpass
from pprint import pprint
import time

# Import the Inference Engine SDK client, authentication helper, and service wrappers.
from teradata_agentstack import BearerAuth
from teradata_agentstack.inference_engine import InferenceEngineClient
from teradata_agentstack.inference_engine import Secrets, Models, Deployments, Jobs, Health

# Import request models used by create, update, rotate, deployment, and resource operations.
from teradata_agentstack.inference_engine.models import (
    SecretCreateRequest,
    SecretUpdateRequest,
    SecretRotateRequest,
    ModelCreateRequest,
    ModelUpdateRequest,
    ModelVersionCreateRequest,
    ModelVersionUpdateRequest,
    DeploymentCreateRequest,
    DeploymentUpdateRequest,
    DeploymentSpec,
    ResourcesConfig,
)


<a id="environment-configuration"></a>
### 1.2 Environment Configuration

This step captures the Inference Engine base URL, workspace, bearer token, and SSL verification behavior for the target environment.


In [ ]:
# Capture environment-specific values used by the Inference Engine client.
BASE_URL = getpass('Enter Inference Core API base URL: ')
WORKSPACE = getpass('Enter Inference workspace: ')
AUTH_TOKEN = getpass('Enter Inference Engine bearer token: ')
print(f'Configured base_url  : {BASE_URL}')
print(f'Configured workspace : {WORKSPACE}')


Token obtained (1577 chars)
InferenceEngine SDK initialized
  base_url  : https://carrot-ai-wrkld.teradata.internal/one-td
  workspace : td-vllm


<a id="authentication"></a>
### 1.3 Authentication

The `teradata_agentstack` SDK supports **4 authentication modes** and **3 ways to provide credentials**.

The credential sources are resolved in this order: direct `auth` parameter, environment variables, then YAML config file.

#### Authentication Modes

| Auth Mode | Class | Required Fields |
| --- | --- | --- |
| Bearer Token | `BearerAuth` | `auth_bearer` |
| Basic Auth | `BasicAuth` | `username`, `password` |
| Client Credentials (OAuth2) | `ClientCredentialsAuth` | `auth_token_url`, `auth_client_id`, `auth_client_secret` |
| Device Code (OAuth2) | `DeviceCodeAuth` | `auth_token_url`, `auth_client_id`, `auth_client_secret`, `auth_device_auth_url` |

This notebook uses `BearerAuth` for Inference Engine requests. Paste a bearer token issued for your target environment; if your environment uses Keycloak, VCE, AI Factory, or another identity flow, obtain the token through that environment's approved login flow before continuing.


In [ ]:
# Create the bearer-auth object used by the Inference Engine client in this notebook.
auth = BearerAuth(auth_bearer=AUTH_TOKEN)
print(f'Bearer token captured ({len(AUTH_TOKEN)} chars)')


<a id="client-initialization"></a>
### 1.4 Client Initialization

This step initializes the Inference Engine SDK client and the service wrappers used by the rest of the notebook.


In [ ]:
# Initialize the Inference Engine SDK client.
client = InferenceEngineClient(
    base_url=BASE_URL,
    auth=auth,
    ssl_verify=SSL_VERIFY,
    workspace=WORKSPACE,
)

# Create service wrappers used by the rest of the notebook.
secrets = Secrets(client=client)
models = Models(client=client)
deployments = Deployments(client=client)
jobs = Jobs(client=client)
health = Health(client=client)

print('InferenceEngine SDK initialized')
print(f'  base_url  : {BASE_URL}')
print(f'  workspace : {WORKSPACE}')


<a id="reusable-helpers"></a>
### 1.5 Reusable Helpers

This step defines the helper used throughout the notebook to display SDK responses consistently.


In [ ]:
# Print SDK responses consistently, whether they are models, dicts, or plain values.
def show_output(label, value):
    print(f'\n{label}:')
    try:
        if hasattr(value, 'model_dump') and callable(value.model_dump):
            pprint(value.model_dump(), sort_dicts=False, width=120)
        elif hasattr(value, 'dict') and callable(value.dict):
            pprint(value.dict(), sort_dicts=False, width=120)
        else:
            pprint(value, sort_dicts=False, width=120)
    except Exception:
        pprint(value, sort_dicts=False, width=120)


<a id="secrets"></a>
## 2. Secrets APIs

Use the Inference Engine SDK to validate the current workspace secret inventory, create a demo HuggingFace secret, then inspect, update, rotate, and delete it through isolated cells.


### 2.1 List Workspace Secrets

This step confirms the workspace is reachable and prints the current secret inventory before any secret mutation is attempted.


In [14]:
# List all secrets in the workspace (paginated)
# Response: SecretListResponse
secret_list = secrets.list(limit=50, offset=0)
show_output('List Secrets', secret_list)



List Secrets:
{'secrets': [{'id': 'b349e642-f746-40f4-92f6-1abb5df16eaf',
              'workspace': 'td-vllm',
              'name': 'hf-secret',
              'type': 'huggingface',
              'k8sSecretName': 'hf-secret',
              'k8sNamespace': 'td-vllm',
              'description': 'HuggingFace token for model downloads',
              'metadata': None,
              'createdAt': '2026-05-14T15:50:03Z',
              'updatedAt': '2026-05-14T15:50:03Z',
              'lastUsedAt': None,
              'createdBy': 'user@example.com',
              'links': {'self': '/v1/workspaces/td-vllm/secrets/hf-secret'}}],
 'pagination': {'total': 1, 'limit': 50, 'offset': 0, 'hasMore': False}}


### 2.2 Create a HuggingFace Secret

This step builds a `SecretCreateRequest`, creates the demo secret, and captures its name for the follow-up secret operations.


In [15]:
# Build and create a HuggingFace secret.
create_secret_body = SecretCreateRequest(
    name='aanchal-demo',
    type='huggingface',            # ngc | huggingface | custom | docker
    value='',                      # Supply your HuggingFace token here
    description='HuggingFace token for demo',
)

created_secret = secrets.create(body=create_secret_body)
show_output('Created Secret', created_secret)

SECRET_NAME = created_secret.name
print(f'SECRET_NAME: {SECRET_NAME}')



Created Secret:
{'id': '816b9d8c-bc9b-4249-b374-f83d380d61a4',
 'workspace': 'td-vllm',
 'name': 'aanchal-demo',
 'type': 'huggingface',
 'k8sSecretName': 'aanchal-demo',
 'k8sNamespace': 'td-vllm',
 'description': 'HuggingFace token for demo',
 'metadata': None,
 'createdAt': '2026-06-22T10:23:27Z',
 'updatedAt': '2026-06-22T10:23:27Z',
 'lastUsedAt': None,
 'createdBy': 'user@example.com',
 'links': {'self': '/v1/workspaces/td-vllm/secrets/aanchal-demo'}}
SECRET_NAME: aanchal-demo


### 2.3 Inspect, Update, Rotate, and Delete Secrets

These cells fetch the created secret, update its metadata/value, rotate the value, and then delete the demo secret when validation is complete.


In [16]:
# Fetch the selected secret by name.
secret_detail = secrets.get(name=SECRET_NAME)
show_output('Get Secret', secret_detail)



Get Secret:
{'id': '816b9d8c-bc9b-4249-b374-f83d380d61a4',
 'workspace': 'td-vllm',
 'name': 'aanchal-demo',
 'type': 'huggingface',
 'k8sSecretName': 'aanchal-demo',
 'k8sNamespace': 'td-vllm',
 'description': 'HuggingFace token for demo',
 'metadata': None,
 'createdAt': '2026-06-22T10:23:27Z',
 'updatedAt': '2026-06-22T10:23:27Z',
 'lastUsedAt': None,
 'createdBy': 'user@example.com',
 'links': {'self': '/v1/workspaces/td-vllm/secrets/aanchal-demo'},
 'usageCount': 0,
 'lastUsedBy': None}


In [17]:
# Build and apply a secret update.
update_secret_body = SecretUpdateRequest(
    value='hf_NEW_TOKEN_HERE',
    description='Updated HuggingFace token',
)

updated_secret = secrets.update(name=SECRET_NAME, body=update_secret_body)
show_output('Updated Secret', updated_secret)



Updated Secret:
{'id': '816b9d8c-bc9b-4249-b374-f83d380d61a4',
 'workspace': 'td-vllm',
 'name': 'aanchal-demo',
 'type': 'huggingface',
 'k8sSecretName': 'aanchal-demo',
 'k8sNamespace': 'td-vllm',
 'description': 'Updated HuggingFace token',
 'metadata': None,
 'createdAt': '2026-06-22T10:23:27Z',
 'updatedAt': '2026-06-22T10:23:36Z',
 'lastUsedAt': None,
 'createdBy': 'user@example.com',
 'links': {'self': '/v1/workspaces/td-vllm/secrets/aanchal-demo'}}


In [18]:
# Build and apply a secret rotation request.
rotate_body = SecretRotateRequest(
    new_value='hf_ROTATED_TOKEN_HERE',
)

rotated_secret = secrets.rotate(name=SECRET_NAME, body=rotate_body)
show_output('Rotated Secret', rotated_secret)



Rotated Secret:
{'id': '816b9d8c-bc9b-4249-b374-f83d380d61a4',
 'workspace': 'td-vllm',
 'name': 'aanchal-demo',
 'type': 'huggingface',
 'k8sSecretName': 'aanchal-demo',
 'k8sNamespace': 'td-vllm',
 'message': "Secret 'aanchal-demo' rotated successfully",
 'rotatedAt': '2026-06-22T10:23:40Z',
 'rotationCount': 1,
 'metadata': None,
 'links': {'self': '/v1/workspaces/td-vllm/secrets/aanchal-demo'}}


In [19]:
# Delete the selected secret from the workspace.
delete_secret_result = secrets.delete(name=SECRET_NAME)
show_output('Delete Secret', delete_secret_result)



Delete Secret:
{'message': "Secret 'aanchal-demo' deleted successfully",
 'secretId': '816b9d8c-bc9b-4249-b374-f83d380d61a4',
 'workspace': 'td-vllm',
 'name': 'aanchal-demo',
 'k8sSecretName': 'aanchal-demo',
 'deletedAt': '2026-06-22T10:23:42Z'}


<a id="models"></a>
## 3. Model Catalog APIs

Use the model catalog SDK to list existing models, create a model download request, inspect model metadata, update catalog fields, and manage versioned model weights.


### 3.1 List Workspace Models, Create a Model, and Poll Download

These cells print the current model catalog, submit a HuggingFace model download request, capture the returned model ID, and poll the async download job before deployment is attempted.

`models.create(..., operation='download')` creates the model catalog record and starts the model-weight download job. There is no separate SDK download call in this workflow; deployment happens later with `deployments.create(...)` after the selected model version is available.


In [20]:
# List all models in the workspace
# Response: ModelListResponse
model_list = models.list(limit=50, offset=0)
show_output('List Models', model_list)



List Models:
{'models': [{'id': 'e69d9e9d-d6f8-4a0b-954e-e64274df055c',
             'name': 'novasearch-stella-en-1-5b-v5',
             'workspace': 'td-vllm',
             'modelType': 'embedding',
             'architecture': 'transformer',
             'status': 'ready',
             'defaultVersion': 'v1',
             'versionCount': 1,
             'labels': {'vendor': 'huggingface'},
             'createdAt': '2026-06-17T11:52:47.231017Z',
             'updatedAt': '2026-06-17T11:53:47.362799Z',
             'links': {'self': '/v1/workspaces/td-vllm/models/e69d9e9d-d6f8-4a0b-954e-e64274df055c',
                       'versions': '/v1/workspaces/td-vllm/models/e69d9e9d-d6f8-4a0b-954e-e64274df055c/versions',
                       'status': None,
                       'metrics': None,
                       'logs': None,
                       'model': None}},
            {'id': '0e6950ae-debd-4bbb-9372-782e661f6c2d',
             'name': 'bge-m3',
             'workspace': 't

In [ ]:
# Build a model download request. models.create(...) creates the catalog record and starts the async download job.
MODEL_SECRET_NAME = None  # Set to a secret name if the HuggingFace model requires credentials.
create_model_body = ModelCreateRequest(
    name='harrier-oss-v1-06c',
    operation='download',                   # download | register
    source_type='huggingface',              # huggingface | ngc | s3 | custom | nim
    source_uri='microsoft/harrier-oss-v1-0.6b',
    model_type='embedding',                 # llm | embedding | reranker | multimodal | custom
    architecture='harrier',
    version='v1.0',
    storage_size='5Gi',
    secret_name=MODEL_SECRET_NAME,          # set to None if model is public
    description='Microsoft Harrier OSS v1 0.6B - small open-source language model',
    labels={'env': 'dev', 'team': 'genai', 'vendor': 'huggingface'},
)

created_model = models.create(body=create_model_body)
show_output('Created Model and Started Download', created_model)

MODEL_ID = created_model.id
print(f'MODEL_ID: {MODEL_ID}')



Created Model:
{'id': '3d21f23b-6e94-416a-b1ec-a9be8a950dfa',
 'name': 'harrier-oss-v1-06c',
 'workspace': 'td-vllm',
 'modelType': 'embedding',
 'architecture': 'harrier',
 'status': 'downloading',
 'jobId': '28658017-dd28-4513-98fa-6b7c738f96a2',
 'version': 'v1.0',
 'versionId': 'd81def30-69a6-4f94-9d74-586083310f2a',
 'message': "Model 'harrier-oss-v1-06c' created. K8s Job 'hf-download-harrier-oss-v1-06c' started downloading "
            "'microsoft/harrier-oss-v1-0.6b' into PVC 'hf-harrier-oss-v1-06c-pvc'.",
 'createdAt': '2026-06-22T10:25:03.516369Z',
 'links': {'self': '/v1/workspaces/td-vllm/models/3d21f23b-6e94-416a-b1ec-a9be8a950dfa',
           'versions': '/v1/workspaces/td-vllm/models/3d21f23b-6e94-416a-b1ec-a9be8a950dfa/versions',
           'status': None,
           'metrics': None,
           'logs': None,
           'model': None}}
MODEL_ID: 3d21f23b-6e94-416a-b1ec-a9be8a950dfa


In [ ]:
# Locate and poll the async model download job started by models.create(... operation='download').
job_list = jobs.list(limit=50, offset=0)
MODEL_JOB_ID = None

for job in job_list.items:
    if job.modelId == MODEL_ID:
        MODEL_JOB_ID = job.id
        break

if MODEL_JOB_ID is None:
    raise RuntimeError(f'No download job found for MODEL_ID {MODEL_ID}')

print(f'MODEL_JOB_ID: {MODEL_JOB_ID}')
terminal_states = {'Succeeded', 'Failed', 'Error', 'Completed'}
successful_states = {'Succeeded', 'Completed'}

for attempt in range(20):
    model_job_result = jobs.get(id=MODEL_JOB_ID)
    print(f'  [{attempt + 1}/20] status: {model_job_result.status}')
    if model_job_result.status in terminal_states:
        break
    time.sleep(30)

show_output('Final Model Download Job State', model_job_result)

if model_job_result.status not in successful_states:
    raise RuntimeError(f'Model download did not complete successfully: {model_job_result.status}')


### 3.2 Inspect and Update the Selected Model

These cells fetch the created model record, display its current catalog state, and apply a metadata update so the update workflow is visible in isolation.


In [27]:
# Fetch the selected model, including non-deleted versions.
model_detail = models.get(id=MODEL_ID)
show_output('Get Model', model_detail)



Get Model:
{'id': '3d21f23b-6e94-416a-b1ec-a9be8a950dfa',
 'name': 'harrier-oss-v1-06c',
 'workspace': 'td-vllm',
 'modelType': 'embedding',
 'architecture': 'harrier',
 'status': 'downloading',
 'defaultVersion': 'v1.0',
 'description': 'Microsoft Harrier OSS v1 0.6B - small open-source language model',
 'metadata': None,
 'labels': {'env': 'dev', 'team': 'genai', 'vendor': 'huggingface'},
 'versions': [{'versionId': 'd81def30-69a6-4f94-9d74-586083310f2a',
               'version': 'v1.0',
               'status': 'downloading',
               'isDefault': True,
               'sourceType': 'huggingface',
               'sourceUri': 'microsoft/harrier-oss-v1-0.6b',
               'storage': {'pvcName': 'hf-harrier-oss-v1-06c-pvc',
                           'storagePath': '/models/harrier-oss-v1-06c/v1.0',
                           'storageSize': '5Gi',
                           'actualSizeBytes': None},
               'jobId': '28658017-dd28-4513-98fa-6b7c738f96a2',
              

In [28]:
# Build and apply a model metadata update.
update_model_body = ModelUpdateRequest(
    description='Updated Harrier OSS description',
    labels={'env': 'staging'},
)

updated_model = models.update(id=MODEL_ID, body=update_model_body)
show_output('Updated Model', updated_model)



Updated Model:
{'id': '3d21f23b-6e94-416a-b1ec-a9be8a950dfa',
 'name': 'harrier-oss-v1-06c',
 'workspace': 'td-vllm',
 'message': 'Model metadata updated successfully.',
 'updatedAt': '2026-06-22T10:25:09.979594Z',
 'links': {'self': '/v1/workspaces/td-vllm/models/3d21f23b-6e94-416a-b1ec-a9be8a950dfa',
           'versions': '/v1/workspaces/td-vllm/models/3d21f23b-6e94-416a-b1ec-a9be8a950dfa/versions',
           'status': None,
           'metrics': None,
           'logs': None,
           'model': None}}


<a id="model-versions"></a>
### 3.3 Manage Model Versions

Each model can have multiple versioned weight sets stored on separate PVCs. Use these cells to list versions, create a new version, inspect it, update metadata/default behavior, and delete it when finished.


In [29]:
# List all versions for the selected model.
version_list = models.list_versions(id=MODEL_ID, limit=50, offset=0)
show_output('List Model Versions', version_list)



List Model Versions:
{'modelId': '3d21f23b-6e94-416a-b1ec-a9be8a950dfa',
 'versions': [{'versionId': 'd81def30-69a6-4f94-9d74-586083310f2a',
               'version': 'v1.0',
               'status': 'downloading',
               'isDefault': True,
               'sourceType': 'huggingface',
               'sourceUri': 'microsoft/harrier-oss-v1-0.6b',
               'storage': {'pvcName': 'hf-harrier-oss-v1-06c-pvc',
                           'storagePath': '/models/harrier-oss-v1-06c/v1.0',
                           'storageSize': '5Gi',
                           'actualSizeBytes': None},
               'jobId': '28658017-dd28-4513-98fa-6b7c738f96a2',
               'createdAt': '2026-06-22T10:25:03.118069Z',
               'updatedAt': '2026-06-22T10:25:03.153801Z'}],
 'pagination': {'total': 1, 'limit': 50, 'offset': 0, 'hasMore': False}}


In [30]:
# Build and create a new model version.
create_version_body = ModelVersionCreateRequest(
    operation='download',
    source_type='huggingface',
    source_uri='meta-llama/Llama-2-7b-chat-hf',
    version='v1.1',
    storage_size='20Gi',
    secret_name=SECRET_NAME,
    set_as_default=False,
)

created_version = models.create_version(id=MODEL_ID, body=create_version_body)
show_output('Created Version', created_version)

VERSION_LABEL = created_version.version
print(f'VERSION_LABEL: {VERSION_LABEL}')



Created Version:
{'modelId': '3d21f23b-6e94-416a-b1ec-a9be8a950dfa',
 'id': '520edecf-3e7c-499d-b7fc-afb1534bdad0',
 'version': 'v1.1',
 'status': 'downloading',
 'jobId': 'e6ea9dbf-13e5-4bea-815a-15ecb22436e1',
 'message': "K8s Job 'hf-download-harrier-oss-v1-06c-v1-1' started downloading 'meta-llama/Llama-2-7b-chat-hf' into "
            "PVC 'hf-harrier-oss-v1-06c-pvc'.",
 'isDefault': False,
 'storage': {'pvcName': 'hf-harrier-oss-v1-06c-pvc',
             'storagePath': '/models/harrier-oss-v1-06c/v1.1',
             'storageSize': '20Gi',
             'actualSizeBytes': None},
 'createdAt': '2026-06-22T10:25:16.142108Z',
 'links': {'self': '/v1/workspaces/td-vllm/models/3d21f23b-6e94-416a-b1ec-a9be8a950dfa/versions/v1.1',
           'versions': None,
           'status': None,
           'metrics': None,
           'logs': None,
           'model': None}}
VERSION_LABEL: v1.1


In [31]:
# Fetch a specific model version and its job/storage details.
version_detail = models.get_version(id=MODEL_ID, version=VERSION_LABEL)
show_output('Get Version', version_detail)



Get Version:
{'modelId': '3d21f23b-6e94-416a-b1ec-a9be8a950dfa',
 'id': '520edecf-3e7c-499d-b7fc-afb1534bdad0',
 'version': 'v1.1',
 'status': 'downloading',
 'jobId': 'e6ea9dbf-13e5-4bea-815a-15ecb22436e1',
 'message': 'Version retrieved successfully.',
 'isDefault': False,
 'storage': {'pvcName': 'hf-harrier-oss-v1-06c-pvc',
             'storagePath': '/models/harrier-oss-v1-06c/v1.1',
             'storageSize': '20Gi',
             'actualSizeBytes': None},
 'createdAt': '2026-06-22T10:25:16.137999Z',
 'links': {'self': '/v1/workspaces/td-vllm/models/3d21f23b-6e94-416a-b1ec-a9be8a950dfa/versions/v1.1',
           'versions': None,
           'status': None,
           'metrics': None,
           'logs': None,
           'model': None}}


In [32]:
# Build and apply a model-version update.
update_version_body = ModelVersionUpdateRequest(
    set_as_default=True,
    metadata={'quantization': 'fp16'},
)

updated_version = models.update_version(id=MODEL_ID, version=VERSION_LABEL, body=update_version_body)
show_output('Updated Version', updated_version)



Updated Version:
{'modelId': '3d21f23b-6e94-416a-b1ec-a9be8a950dfa',
 'id': '520edecf-3e7c-499d-b7fc-afb1534bdad0',
 'version': 'v1.1',
 'status': 'downloading',
 'jobId': 'e6ea9dbf-13e5-4bea-815a-15ecb22436e1',
 'message': 'Version updated successfully.',
 'isDefault': True,
 'storage': {'pvcName': 'hf-harrier-oss-v1-06c-pvc',
             'storagePath': '/models/harrier-oss-v1-06c/v1.1',
             'storageSize': '20Gi',
             'actualSizeBytes': None},
 'createdAt': '2026-06-22T10:25:16.137999Z',
 'links': {'self': '/v1/workspaces/td-vllm/models/3d21f23b-6e94-416a-b1ec-a9be8a950dfa/versions/v1.1',
           'versions': None,
           'status': None,
           'metrics': None,
           'logs': None,
           'model': None}}


In [33]:
# Delete the selected model version.
delete_version_result = models.delete_version(id=MODEL_ID, version=VERSION_LABEL)
show_output('Delete Version', delete_version_result)



Delete Version:
{'message': "Version 'v1.1' deleted. K8s Job and PVC 'hf-harrier-oss-v1-06c-pvc' have been removed.",
 'modelId': '3d21f23b-6e94-416a-b1ec-a9be8a950dfa',
 'versionId': '520edecf-3e7c-499d-b7fc-afb1534bdad0',
 'version': 'v1.1',
 'workspace': 'td-vllm',
 'deletedAt': '2026-06-22T10:25:26.301746Z'}


<a id="deployments"></a>
## 4. Deployment APIs

Deploy models as Kubernetes inference workloads, inspect deployment state, update runtime settings, and delete demo deployments through direct SDK calls.


### 4.1 List and Create Deployments

These cells show the current deployment inventory, then create a KServe deployment for the model version downloaded in Section 3.

`deployments.create(...)` creates the serving workload only. It does not download model weights, so run this section after the model download job has completed successfully.


In [35]:
# List all deployments in the workspace
# Response: DeploymentListResponse
deployment_list = deployments.list(limit=50, offset=0)
show_output('List Deployments', deployment_list)



List Deployments:
{'deployments': [{'workloadId': 'c883c004-ff2e-438f-916e-faef581a0cbc',
                  'name': 'qwen-3-5-122b-a10b-gptq-int4',
                  'modelId': '4cf44fa4-5a74-4e7e-b301-84c2e8e8b56b',
                  'workspace': 'td-vllm',
                  'phase': 'Running',
                  'replicas': 1,
                  'endpointUrl': 'https://carrot-ai-wrkld.teradata.internal/one-td/litellm/v1/chat/completions',
                  'createdAt': '2026-06-16T20:40:26.634167+00:00',
                  'updatedAt': '2026-06-16T20:49:00.433058+00:00',
                  'links': {'self': '/v1/workspaces/td-vllm/deployments/c883c004-ff2e-438f-916e-faef581a0cbc',
                            'versions': None,
                            'status': '/v1/workspaces/td-vllm/deployments/c883c004-ff2e-438f-916e-faef581a0cbc/status',
                            'metrics': None,
                            'logs': None,
                            'model': None}},
             

In [ ]:
# Build a deployment request for the model version downloaded in Section 3.
create_deployment_body = DeploymentCreateRequest(
    name='harrier-oss-v1-06b-deploy',
    model_id=MODEL_ID,
    version='v1.0',
    engine='kserve',
    runtime='vllm',
    replicas=1,
    spec=DeploymentSpec(
        resources=ResourcesConfig(
            gpu_type='RTX-PRO-6000',
            gpu_limit=1,
            gpu_request=1,
            memory_limit='16Gi',
            memory_request='8Gi',
            cpu_limit='8',
            cpu_request='4',
        ),
    ),
)

created_deployment = deployments.create(body=create_deployment_body)
show_output('Created Deployment', created_deployment)

DEPLOYMENT_ID = created_deployment.workloadId
print(f'DEPLOYMENT_ID: {DEPLOYMENT_ID}')


### 4.2 Inspect Deployment Status

These cells fetch the created deployment record and use the deployment detail response as the status view for workload phase, endpoint URL, and LiteLLM model ID.


In [ ]:
# Fetch the deployment details, including endpoint URL and workload phase.
deployment_detail = deployments.get(id=DEPLOYMENT_ID)
show_output('Get Deployment', deployment_detail)


In [ ]:
# Use deployment details as the deployment status view.
deployment_status = deployments.get(id=DEPLOYMENT_ID)
show_output('Deployment Details (Status)', deployment_status)


### 4.3 Update and Delete Deployments

These cells apply a deployment update request and then delete the demo deployment so the full deployment lifecycle is represented.


In [ ]:
# Build and apply a deployment update.
update_deployment_body = DeploymentUpdateRequest(
    replicas=2,
)

updated_deployment = deployments.update(id=DEPLOYMENT_ID, body=update_deployment_body)
show_output('Updated Deployment', updated_deployment)


In [ ]:
# Delete the selected deployment.
delete_deployment_result = deployments.delete(id=DEPLOYMENT_ID, force=False)
show_output('Delete Deployment', delete_deployment_result)


<a id="jobs"></a>
## 5. Job Monitoring APIs

Jobs track async Kubernetes workloads such as model download jobs and NIMCache CRDs. Use these cells to list, sync, and inspect job status while resources are being prepared.


### 5.1 List and Sync Jobs

These cells print current workspace jobs and trigger a sync for non-terminal jobs so the notebook can show refreshed job state from the live cluster.


In [43]:
# List all jobs in the workspace with optional filters
# Response: JobListResponse
job_list = jobs.list(limit=50, offset=0)
show_output('List Jobs', job_list)



List Jobs:
{'data': [{'jobId': 'e6ea9dbf-13e5-4bea-815a-15ecb22436e1',
           'jobType': 'k8s_job',
           'modelId': '3d21f23b-6e94-416a-b1ec-a9be8a950dfa',
           'modelName': 'harrier-oss-v1-06c',
           'version': 'v1.1',
           'operation': 'download',
           'status': 'running',
           'progress': {'percent': 0, 'downloadedBytes': None, 'totalBytes': None, 'message': 'In progress'},
           'pvcName': 'hf-harrier-oss-v1-06c-pvc',
           'nimcacheStatus': None,
           'createdAt': '2026-06-22T10:25:16.170831+00:00',
           'updatedAt': '2026-06-22T10:25:16.170831+00:00'},
          {'jobId': '28658017-dd28-4513-98fa-6b7c738f96a2',
           'jobType': 'k8s_job',
           'modelId': '3d21f23b-6e94-416a-b1ec-a9be8a950dfa',
           'modelName': 'harrier-oss-v1-06c',
           'version': 'v1.0',
           'operation': 'download',
           'status': 'running',
           'progress': {'percent': 0, 'downloadedBytes': None, 'totalByte

In [44]:
# Sync non-terminal job status from the live Kubernetes cluster.
sync_result = jobs.sync()
show_output('Sync Jobs', sync_result)



Sync Jobs:
{'workspace': 'td-vllm',
 'jobsChecked': 2,
 'jobsUpdated': 0,
 'results': [{'jobId': '28658017-dd28-4513-98fa-6b7c738f96a2',
              'jobType': 'k8s_job',
              'previousStatus': 'running',
              'newStatus': 'running',
              'modelId': '3d21f23b-6e94-416a-b1ec-a9be8a950dfa',
              'modelUpdated': False,
              'versionUpdated': False,
              'changed': False},
             {'jobId': 'e6ea9dbf-13e5-4bea-815a-15ecb22436e1',
              'jobType': 'k8s_job',
              'previousStatus': 'running',
              'newStatus': 'running',
              'modelId': '3d21f23b-6e94-416a-b1ec-a9be8a950dfa',
              'modelUpdated': False,
              'versionUpdated': False,
              'changed': False}]}


### 5.2 Inspect a Specific Job

This step fetches a single job by ID so you can inspect detailed status, progress, and any error message returned by the SDK.


In [ ]:
# Fetch a specific job by ID.
JOB_ID = ''  # Set to a job ID from the list above
job_detail = jobs.get(id=JOB_ID)
show_output('Get Job', job_detail)


<a id="health"></a>
## 6. Health APIs

Run the service health, liveness, and readiness checks so the notebook shows the current operational state of the Inference Engine API.


### 6.1 Run Service Health Checks

These cells call health, liveness, and readiness endpoints so the notebook shows service availability and dependency readiness.


In [46]:
# Basic health check (liveness probe) - checks if the service is running and responsive.
# Response: HealthCheckResponse
show_output('Health Check', health.health_check())



Health Check:
{'status': 'healthy',
 'service': 'inference-core-service',
 'version': '1.0.0',
 'timestamp': '2026-06-22T10:28:05.103836Z'}


In [47]:
# Liveness probe - Kubernetes liveness check.
# Returns 200 if the application is alive; Kubernetes will restart the pod if this fails.
show_output('Liveness', health.liveness())



Liveness:
{'status': 'alive', 'service': 'inference-core-service', 'timestamp': '2026-06-22T10:28:08.015124Z'}


In [48]:
# Readiness probe - Kubernetes readiness check.
# Verifies database connectivity and critical dependencies.
# Returns 200 if ready to serve traffic, 503 if not ready.
show_output('Readiness', health.readiness())



Readiness:
{'service': 'inference-core-service',
 'version': '1.0.0',
 'timestamp': '2026-06-22T10:28:09.774723Z',
 'status': 'ready',
 'checks': {'database': {'status': 'healthy'}}}


<a id="s3-model-workflow"></a>
## 7. S3 Model Download and Deployment

Download a pre-packaged model from the internal S3 bucket, poll the async download job, deploy the resulting model version, inspect deployment state, and clean up the S3 demo resources.

> **Note:** Set `source_type='s3'` to trigger an async S3 download job that places model weights on a PVC. Point `s3Config` to the bucket and key prefix where the weights are stored. The internal bucket does not require credentials, so `secret_name=None` is used.


### 7.1 Configure S3 Model Metadata

This step defines the model name, S3 bucket, key prefix, model type, architecture, version label, and storage request used by the S3 download workflow.


In [52]:
# Model: Harrier OSS v1 0.6B (Microsoft) sourced from the internal S3 bucket.
S3_MODEL_NAME = 'harrier-oss-v1-06c'
S3_MODEL_DESCRIPTION = 'Microsoft Harrier OSS v1 0.6B - small open-source embedding model'
S3_SOURCE_URI = 'hf-models/microsoft__harrier-oss-v1-0.6b'
S3_BUCKET = 'inference-models-bucket'
S3_KEY_PREFIX = 'hf-models/microsoft__harrier-oss-v1-0.6b'
S3_MODEL_TYPE = 'embedding'      # llm | embedding | reranker | multimodal | custom
S3_ARCHITECTURE = 'harrier'
S3_VERSION = 'v1'
S3_STORAGE_SIZE = '5Gi'

print(f'S3 model target : s3://{S3_BUCKET}/{S3_KEY_PREFIX}')
print(f'Model name      : {S3_MODEL_NAME}')


S3 model target : s3://inference-models-bucket/hf-models/microsoft__harrier-oss-v1-0.6b
Model name      : harrier-oss-v1-06c


### 7.2 Create and Poll the S3 Model Download

These cells submit the S3-backed model download request, capture the returned model ID, locate the async job, and poll until it reaches a terminal state or the polling limit is reached.


In [53]:
# Build and create an S3-backed model download request.
s3_create_model_body = ModelCreateRequest(
    name=S3_MODEL_NAME,
    operation='download',
    source_type='s3',
    source_uri=S3_SOURCE_URI,
    model_type=S3_MODEL_TYPE,
    architecture=S3_ARCHITECTURE,
    version=S3_VERSION,
    storage_size=S3_STORAGE_SIZE,
    secret_name=None,           # internal S3 bucket - no credentials needed
    description=S3_MODEL_DESCRIPTION,
    labels={'vendor': 'microsoft', 'source': 's3', 'task': 'embedding'},
    s3Config={
        'bucket': S3_BUCKET,
        'keyPrefix': S3_KEY_PREFIX,
        'downloadSource': 'huggingface',
    },
)

s3_created_model = models.create(body=s3_create_model_body)
show_output('S3 Created Model', s3_created_model)

S3_MODEL_ID = s3_created_model.id
print(f'S3_MODEL_ID: {S3_MODEL_ID}')



S3 Created Model:
{'id': 'a6161491-39bd-45da-a48f-91d7771dc962',
 'name': 'harrier-oss-v1-06c',
 'workspace': 'td-vllm',
 'modelType': 'embedding',
 'architecture': 'harrier',
 'status': 'downloading',
 'jobId': 'f75d615e-3394-4fe6-a955-a0a3b05de2df',
 'version': 'v1',
 'versionId': '3668d665-2426-4cee-8549-8215cdef52c9',
 'message': "Model 'harrier-oss-v1-06c' created. K8s Job 's3-download-harrier-oss-v1-06c' started downloading "
            "'hf-models/microsoft__harrier-oss-v1-0.6b' into PVC 's3-harrier-oss-v1-06c-pvc'.",
 'createdAt': '2026-06-22T10:28:50.254687Z',
 'links': {'self': '/v1/workspaces/td-vllm/models/a6161491-39bd-45da-a48f-91d7771dc962',
           'versions': '/v1/workspaces/td-vllm/models/a6161491-39bd-45da-a48f-91d7771dc962/versions',
           'status': None,
           'metrics': None,
           'logs': None,
           'model': None}}
S3_MODEL_ID: a6161491-39bd-45da-a48f-91d7771dc962


In [ ]:
# Find and poll the async S3 model download job.
job_list = jobs.list(limit=50, offset=0)
S3_JOB_ID = None

for job in job_list.items:
    if job.modelId == S3_MODEL_ID:
        S3_JOB_ID = job.id
        break

print(f'S3_JOB_ID: {S3_JOB_ID}')
terminal_states = {'Succeeded', 'Failed', 'Error', 'Completed'}

for attempt in range(20):
    s3_job_result = jobs.get(id=S3_JOB_ID)
    print(f'  [{attempt + 1}/20] status: {s3_job_result.status}')
    if s3_job_result.status in terminal_states:
        break
    time.sleep(30)

show_output('Final S3 Job State', s3_job_result)


### 7.3 Deploy and Inspect the S3 Model

These cells create a KServe deployment for the downloaded S3 model, capture the deployment workload ID, and fetch deployment state for inspection.


In [55]:
# Build and create a deployment for the S3-backed model.
s3_deployment_body = DeploymentCreateRequest(
    name=f'{S3_MODEL_NAME}-deploy',
    model_id=S3_MODEL_ID,
    version=S3_VERSION,
    engine='kserve',
    runtime='vllm',
    replicas=1,
    spec=DeploymentSpec(
        resources=ResourcesConfig(
            gpu_type='RTX-PRO-6000',
            gpu_limit=1,
            gpu_request=1,
            memory_limit='16Gi',
            memory_request='8Gi',
            cpu_limit='8',
            cpu_request='4',
        ),
    ),
    runtime_args={
        'tensor_parallel_size': 1,
        'dtype': 'auto',
        'enable_prefix_caching': False,
        'trust_remote_code': True,
        'runner': 'pooling',    # required for embedding models
    },
)

s3_created_deployment = deployments.create(body=s3_deployment_body)
show_output('S3 Created Deployment', s3_created_deployment)

S3_DEPLOYMENT_ID = s3_created_deployment.workloadId
print(f'S3_DEPLOYMENT_ID: {S3_DEPLOYMENT_ID}')



S3 Created Deployment:
{'workloadId': '18ad137a-bb86-4579-b2bb-6780eebfccdb',
 'name': 'harrier-oss-v1-06c-deploy',
 'modelId': 'a6161491-39bd-45da-a48f-91d7771dc962',
 'version': 'v1',
 'engine': 'kserve',
 'runtime': 'vllm',
 'replicas': 1,
 'workspace': 'td-vllm',
 'phase': 'Creating',
 'message': 'Workload submission initiated — InferenceService pending',
 'spec': {'resources': {'gpuType': 'RTX-PRO-6000',
                        'gpuLimit': 1,
                        'gpuRequest': 1,
                        'memoryLimit': '16Gi',
                        'memoryRequest': '8Gi',
                        'cpuLimit': '8',
                        'cpuRequest': '4'},
          'autoscaling': None,
          'runtimeConfig': None,
          'annotations': None,
          'labels': None,
          'scheduling': None},
 'endpointUrl': 'https://carrot-ai-wrkld.teradata.internal/one-td/litellm/v1/embeddings',
 'inferenceServiceName': 'harrier-oss-v1-06c-deploy',
 'litellmModelId': None,
 'cre

In [56]:
# Fetch deployment status for the S3 deployment.
s3_deployment_status = deployments.get(id=S3_DEPLOYMENT_ID)
show_output('S3 Deployment Status', s3_deployment_status)



S3 Deployment Status:
{'workloadId': '18ad137a-bb86-4579-b2bb-6780eebfccdb',
 'name': 'harrier-oss-v1-06c-deploy',
 'modelId': 'a6161491-39bd-45da-a48f-91d7771dc962',
 'version': 'v1',
 'engine': 'kserve',
 'runtime': 'vllm',
 'replicas': 1,
 'workspace': 'td-vllm',
 'phase': 'Creating',
 'message': '',
 'spec': {'resources': {'gpuType': 'RTX-PRO-6000',
                        'gpuLimit': 1,
                        'gpuRequest': 1,
                        'memoryLimit': '16Gi',
                        'memoryRequest': '8Gi',
                        'cpuLimit': '8',
                        'cpuRequest': '4'},
          'autoscaling': None,
          'runtimeConfig': None,
          'annotations': None,
          'labels': None,
          'scheduling': None},
 'endpointUrl': 'https://carrot-ai-wrkld.teradata.internal/one-td/litellm/v1/embeddings',
 'litellmModelId': None,
 'createdAt': '2026-06-22T10:29:03.880330+00:00',
 'updatedAt': '2026-06-22T10:29:03.880330+00:00',
 'links': {'self

### 7.4 Clean Up S3 Demo Resources

This step deletes the S3 deployment and S3 model created by the S3 workflow so the demo resources do not remain active after validation.


In [ ]:
# Delete the S3 deployment and S3 model resources created above.
s3_delete_deployment_result = deployments.delete(id=S3_DEPLOYMENT_ID, force=False)
show_output('S3 Cleanup - Delete Deployment', s3_delete_deployment_result)

s3_delete_model_result = models.delete(id=S3_MODEL_ID)
show_output('S3 Cleanup - Delete Model', s3_delete_model_result)


<a id="cleanup"></a>
## 8. Cleanup and Results

Delete the demo deployment, model, and secret resources when you are finished, then review the outcome and reuse notes for future runs.


### 8.1 Delete Demo Resources

This step deletes the deployment, model, and secret created by the primary workflow once you are finished inspecting the SDK responses.


In [ ]:
# Delete the deployment created in the deployment section.
cleanup_deployment_result = deployments.delete(id=DEPLOYMENT_ID, force=False)
show_output('Cleanup - Delete Deployment', cleanup_deployment_result)

# Delete the model created in the model section.
cleanup_model_result = models.delete(id=MODEL_ID)
show_output('Cleanup - Delete Model', cleanup_model_result)

# Delete the secret created in the secrets section.
cleanup_secret_result = secrets.delete(name=SECRET_NAME)
show_output('Cleanup - Delete Secret', cleanup_secret_result)


### 8.2 Results and Interpretation

Run the notebook from top to bottom after confirming resource names, IDs, and credentials for your environment. Each code cell performs the step for that section directly and shows the returned SDK response so the operator can inspect current state before moving on.

**Key operating notes:**
- Review mutating cells such as `create`, `update`, `delete`, `rotate`, and `sync` before running them.
- `InferenceEngineClient` accepts a `workspace` parameter that is injected automatically into every API call. Override it per-call by passing `workspace=...` explicitly.
- `body` parameters accept either a Pydantic model or a plain `dict`.
- Creating a model triggers an async job. Poll `jobs.get(id=job_id)` or call `jobs.sync()` to track download progress before creating a deployment.
- The `litellmModelId` field on `DeploymentResponse` is the model name to use in `/v1/chat/completions` calls once the deployment is `Running`.
- KServe deployments require `engine='kserve'` and `runtime='vllm'`. NIM deployments require `engine='nim'` and optionally a `nim_config` in `ModelCreateRequest`.


### 8.3 Summary / Next Steps

This notebook now uses the same direct execution pattern as the MCP getting-started guide: configure the environment at the top, run each numbered SDK workflow step in order, inspect returned SDK objects after each action, and clean up demo resources at the end.

**Key points:**
- Inputs and environment values are centralized in the setup section.
- Each section uses direct Inference Engine SDK calls without hidden wrapper workflows.
- Status checks use `deployments.get(...)`, `jobs.get(...)`, and health probes so the notebook stays readable and easy to troubleshoot.
- The final cleanup cells remove demo deployments, models, and secrets created by the guide.

**Author:** [Aanchal Kavedia, Vino M Mathew]  
**Version:** 1.0  
**Last Updated:** June 2026  
**Copyright:** 2026 Teradata. All rights reserved.
